# 面试问题：成百上千个 LoRA Adapter 怎样共享一个 Base Model 服务？异构 batching、缓存和隔离如何设计？

**一句话回答**：Base 权重只驻留一份，请求通过不可变 `adapter_id+revision` 选择低秩增量 `ΔW=(α/r)AB`。同一 batch 先做共享 base GEMM，再按 adapter 分组做低秩更新；Adapter 从 CPU/存储分页到 GPU，使用引用计数和 LRU。Registry 必须绑定 base revision、rank、dtype、digest 与 tenant ACL，热更新采用校验—预热—原子发布。

本 Notebook 用 NumPy 手写 LoRA forward、异构 batch 聚合、Adapter Registry、GPU cache、调度、两阶段发布、租户隔离和容量指标。关键路径均配中文注释，不调用 PEFT 或 Serving 框架。


In [ ]:
from collections import OrderedDict  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
import hashlib,math  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

# 小矩阵足以验证共享 base 与异构 adapter 的数值语义。
SEED145=14501; rng145=np.random.default_rng(SEED145)  # 计算并保存当前步骤的中间状态。
assert SEED145==14501  # 用受控断言验证关键不变量。
assert np.isfinite(rng145.normal())  # 用受控断言验证关键不变量。
assert 8//2==4  # 用受控断言验证关键不变量。


## 1. LoRA 服务的外部合同是 base + adapter revision

对输入 `X`，输出 `XW + scale·(XA)B`。不同训练代码可能交换 A/B 命名，但 shape 与 scale 必须固化。adapter 只对训练时的 base checkpoint 有意义，不能凭 hidden size 相同跨 base 复用。


In [ ]:
D145,O145,R145=6,5,2; W145=rng145.normal(size=(D145,O145)); A145=rng145.normal(size=(D145,R145)); B145=rng145.normal(size=(R145,O145))  # 计算并保存当前步骤的中间状态。
def lora_forward145(X,W,A,B,alpha):  # 定义本节可复用的核心函数。
    # 先投影到低秩空间，再映射回输出维；scale 采用 alpha/rank。
    return X@W+(alpha/A.shape[1])*(X@A)@B  # 返回当前分支计算出的结果。
X145=rng145.normal(size=(4,D145)); Y145=lora_forward145(X145,W145,A145,B145,4)  # 计算并保存当前步骤的中间状态。
assert Y145.shape==(4,O145)  # 用受控断言验证关键不变量。
assert np.allclose(lora_forward145(X145,W145,A145*0,B145,4),X145@W145)  # 用受控断言验证关键不变量。
assert not np.allclose(Y145,X145@W145)  # 用受控断言验证关键不变量。


## 2. 异构 batch 共享 base GEMM，再分组计算 LoRA

若逐请求复制完整模型，显存和 batch 利用率都很差。把 batch 中相同 adapter 的行 gather 到一起，计算 `(XA)B` 后 scatter 回原位置；base `XW` 只算一次。生产系统用定制 kernel 融合不同 rank/adapter，而不是 Python 循环。


In [ ]:
adapters145={"a":(A145,B145,4.),"b":(rng145.normal(size=(D145,1)),rng145.normal(size=(1,O145)),2.)}  # 计算并保存当前步骤的中间状态。
ids145=np.array(["a","b","a","b"])  # 计算并保存当前步骤的中间状态。
def heterogeneous145(X,W,ids,adapters):  # 定义本节可复用的核心函数。
    # 共享 base 输出，按 adapter 索引只写对应行的增量。
    out=X@W  # 计算并保存当前步骤的中间状态。
    for aid in sorted(set(ids)):  # 遍历输入元素以累积或检查结果。
        rows=np.where(ids==aid)[0]; A,B,alpha=adapters[aid]; out[rows]+=(alpha/A.shape[1])*(X[rows]@A)@B  # 计算并保存当前步骤的中间状态。
    return out  # 返回当前分支计算出的结果。
hetero145=heterogeneous145(X145,W145,ids145,adapters145); naive145=np.stack([lora_forward145(X145[i:i+1],W145,*adapters145[aid])[0] for i,aid in enumerate(ids145)])  # 计算并保存当前步骤的中间状态。
assert np.allclose(hetero145,naive145)  # 用受控断言验证关键不变量。
assert hetero145.shape==(4,5)  # 用受控断言验证关键不变量。
assert len(set(ids145))==2  # 用受控断言验证关键不变量。


## 3. Registry 拒绝 base、rank、shape 或 digest 不一致

Adapter metadata 至少包含 tenant、adapter revision、base revision、rank、alpha、dtype、shape、训练模板与制品 digest。服务发现名称不等于授权；请求 principal 还要通过 ACL。不可变 revision 便于并发请求稳定和回滚。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class AdapterMeta145:  # 定义承载本节状态与行为的数据结构。
    tenant:str; name:str; revision:str; base_revision:str; rank:int; alpha:float; digest:str  # 执行当前语句以推进本节示例。
    def key(self): return (self.tenant,self.name,self.revision)  # 定义本节可复用的核心函数。
def make_meta145(tenant,name,rev,base,A,B,alpha):  # 定义本节可复用的核心函数。
    # digest 同时覆盖 A/B 原始字节，防止 registry 指向被替换文件。
    digest=hashlib.sha256(A.tobytes()+B.tobytes()).hexdigest(); return AdapterMeta145(tenant,name,rev,base,A.shape[1],alpha,digest)  # 计算并保存当前步骤的中间状态。
meta145=make_meta145("t1","support","v2","base-7",A145,B145,4.)  # 计算并保存当前步骤的中间状态。
assert meta145.rank==R145  # 用受控断言验证关键不变量。
assert len(meta145.digest)==64  # 用受控断言验证关键不变量。
assert meta145.key()==("t1","support","v2")  # 用受控断言验证关键不变量。


## 4. GPU Adapter Cache 用 byte quota、引用计数和可回收 LRU

活跃请求 retain adapter，结束后 release；淘汰只能选择 refcount=0 的条目。若所有条目都在使用，准入应排队或失败，不能覆盖正在被 kernel 读取的权重。不同 rank 的真实字节而非条目数决定容量。


In [ ]:
class AdapterCache145:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,capacity): self.capacity=capacity; self.used=0; self.data=OrderedDict(); self.refs={}  # 定义本节可复用的核心函数。
    def put(self,key,size):  # 定义本节可复用的核心函数。
        # 先淘汰最老且无引用条目，直到新 adapter 能放入。
        while self.used+size>self.capacity:  # 在终止条件满足前持续推进状态。
            victim=next((k for k in self.data if self.refs[k]==0),None)  # 计算并保存当前步骤的中间状态。
            if victim is None: raise RuntimeError("adapter_cache_full")  # 按当前条件选择后续控制路径。
            self.used-=self.data.pop(victim); self.refs.pop(victim)  # 计算并保存当前步骤的中间状态。
        self.data[key]=size; self.refs[key]=0; self.used+=size  # 计算并保存当前步骤的中间状态。
    def retain(self,key): self.refs[key]+=1; self.data.move_to_end(key)  # 定义本节可复用的核心函数。
    def release(self,key): self.refs[key]-=1  # 定义本节可复用的核心函数。
cache145=AdapterCache145(100); cache145.put("a",60); cache145.put("b",30); cache145.retain("a"); cache145.put("c",40)  # 计算并保存当前步骤的中间状态。
assert "a" in cache145.data and "b" not in cache145.data  # 用受控断言验证关键不变量。
assert cache145.used==100  # 用受控断言验证关键不变量。
assert cache145.refs["a"]==1  # 用受控断言验证关键不变量。


## 5. 调度器平衡 base batching、adapter locality 与等待公平

合并更多 adapter 可增大 base batch，却增加 adapter load 与异构 kernel 开销；只追热点又会饿死冷 adapter。优先级可结合 deadline、age、adapter 是否驻留、rank 成本和 KV 预算，并给冷请求 aging bonus。


In [ ]:
requests145=[{"id":"r1","adapter":"a","age":1,"resident":1},{"id":"r2","adapter":"cold","age":20,"resident":0},{"id":"r3","adapter":"a","age":2,"resident":1}]  # 计算并保存当前步骤的中间状态。
def priority145(r):  # 定义本节可复用的核心函数。
    # 驻留奖励提升 locality，age 项保证冷 adapter 最终不会饥饿。
    return 2*r["resident"]+.2*r["age"]  # 返回当前分支计算出的结果。
order145=sorted(requests145,key=priority145,reverse=True)  # 计算并保存当前步骤的中间状态。
assert order145[0]["id"]=="r2"  # 用受控断言验证关键不变量。
assert priority145(requests145[0])<priority145(requests145[1])  # 用受控断言验证关键不变量。
assert {r["id"] for r in order145}=={"r1","r2","r3"}  # 用受控断言验证关键不变量。


## 6. 热加载采用 staging、数值探针和原子发布

下载到 staging 后校验 digest/base/shape/dtype，在固定 probe 上检查输出有限和阈值，再预热 GPU；最后让新请求指向新 revision。旧请求继续持有旧 revision，待 refcount 清零再回收，失败则不改变 active pointer。


In [ ]:
active145={"support":"v1"}  # 计算并保存当前步骤的中间状态。
def publish145(name,revision,meta,A,B,expected_base,probe):  # 定义本节可复用的核心函数。
    # 所有校验成功后才执行最后一行的原子指针切换。
    if meta.base_revision!=expected_base or A.shape[1]!=meta.rank or B.shape[0]!=meta.rank: raise ValueError("adapter_incompatible")  # 按当前条件选择后续控制路径。
    if hashlib.sha256(A.tobytes()+B.tobytes()).hexdigest()!=meta.digest: raise ValueError("digest")  # 按当前条件选择后续控制路径。
    if not np.all(np.isfinite(probe@A@B)): raise ValueError("nonfinite")  # 按当前条件选择后续控制路径。
    active145[name]=revision  # 计算并保存当前步骤的中间状态。
publish145("support","v2",meta145,A145,B145,"base-7",X145[:1])  # 执行当前语句以推进本节示例。
assert active145["support"]=="v2"  # 用受控断言验证关键不变量。
try: publish145("support","bad",meta145,A145[:,:1],B145,"base-7",X145[:1]); raise AssertionError("bad publish")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="adapter_incompatible"  # 捕获预期异常并验证失败分支。
assert active145["support"]=="v2"  # 用受控断言验证关键不变量。


## 7. Adapter、KV Cache 与日志按 tenant 隔离

同名 adapter 在不同 tenant 下是不同 key；principal 必须同时获得 adapter 和 base endpoint 权限。不要把用户输入、adapter 路径或 digest 当授权信息。错误消息和 metrics 也避免泄露其他租户有哪些 adapter 或是否命中缓存。


In [ ]:
acl145={("t1","support","v2"):{"alice"},("t2","support","v2"):{"bob"}}  # 计算并保存当前步骤的中间状态。
def authorize145(principal,key):  # 定义本节可复用的核心函数。
    # Registry key 已包含 tenant，防止只按名称查找串租户。
    if principal not in acl145.get(key,set()): raise PermissionError("adapter_acl")  # 按当前条件选择后续控制路径。
    return True  # 返回当前分支计算出的结果。
assert authorize145("alice",("t1","support","v2"))  # 用受控断言验证关键不变量。
try: authorize145("alice",("t2","support","v2")); raise AssertionError("cross tenant")  # 尝试执行可能失败的受控操作。
except PermissionError as e: assert str(e)=="adapter_acl"  # 捕获预期异常并验证失败分支。
assert ("t1","support","v2")!=("t2","support","v2")  # 用受控断言验证关键不变量。


## 8. 评测 adapter 规模、异构度和冷启动分布

指标包括 active adapters、GPU hit/eviction/load latency、base batch size、每批唯一 adapter 数、rank 分布、TTFT/TPOT、吞吐、公平性和逐 adapter 质量。受控矩阵等价只证明语义，Punica/S-LoRA 的性能来自 paging、scheduler 与专用 kernel。


In [ ]:
def adapter_bytes145(din,dout,rank,bytes_=2):  # 定义本节可复用的核心函数。
    # LoRA 只存 A/B，两矩阵字节与 rank 线性增长。
    return (din*rank+rank*dout)*bytes_  # 返回当前分支计算出的结果。
bytes_r8_145=adapter_bytes145(4096,4096,8); bytes_r64_145=adapter_bytes145(4096,4096,64)  # 计算并保存当前步骤的中间状态。
assert bytes_r64_145==8*bytes_r8_145  # 用受控断言验证关键不变量。
assert bytes_r8_145<4096*4096*2  # 用受控断言验证关键不变量。
assert cache145.used<=cache145.capacity  # 用受控断言验证关键不变量。


## 面试总结

完整主线是：**base 与 adapter revision 绑定 → 验证 `XW+(α/r)(XA)B` → base GEMM 共享、按 adapter gather/scatter → registry 固化 shape/digest/tenant → GPU cache byte quota+refcount+LRU → locality 与 age 联合调度 → staging 校验/预热/原子发布 → ACL 隔离 → 冷启动、异构 batch、逐 adapter 质量与 TTFT/TPOT 联合评测**。

延伸阅读：[Punica](https://arxiv.org/abs/2310.18547)、[S-LoRA](https://arxiv.org/abs/2311.03285)、[LoRA](https://arxiv.org/abs/2106.09685)。
